## Using The AWS Python SDK (Boto3)

You can find a general overview and links to deeper information in the [AWS SDK for Python (Boto3) documentation](https://boto3.amazonaws.com/v1/documentation/api)

Boto3 is your gateway to *all* of the AWS services with Python.

We will need to use boto3 to access our previously stored secret from within our code. 

For this notebook there are a handful logical steps, validating each along the way:
- Install Boto3 into our virtual environment
- Create a session with our AWS credentials
- Create a Secrets Manager client
- Retrieve the secret
- Parse the values and format it for use in our code


In [1]:
# Step One: Install the package
!python3 -m pip install boto3

  Using cached boto3-1.34.39-py3-none-any.whl.metadata (6.6 kB)
  Using cached botocore-1.34.39-py3-none-any.whl.metadata (5.7 kB)
  Using cached jmespath-1.0.1-py3-none-any.whl (20 kB)
  Using cached s3transfer-0.10.0-py3-none-any.whl.metadata (1.7 kB)
  Using cached urllib3-2.0.7-py3-none-any.whl.metadata (6.6 kB)
Using cached boto3-1.34.39-py3-none-any.whl (139 kB)
Using cached botocore-1.34.39-py3-none-any.whl (11.9 MB)
Using cached s3transfer-0.10.0-py3-none-any.whl (82 kB)
Using cached urllib3-2.0.7-py3-none-any.whl (124 kB)


In [2]:
# Validate the install of boto3
# - you should see a version number as an output for this cell - 
# eg. 1.34.39
!python3 -c "import boto3; print(boto3.__version__)"

1.34.39


### Boto3 Authentication
Boto3 uses the credentials we set up earlier for our IAM account (*secrets-user*) - those credentials located in the *~/.aws/credentials* file.

If you are uncertain of its existence, you can run the following terminal command in the notebook to check:

`!ls ~/.aws/credentials`

and to print the values to the output, you can run the following command:

`!cat ~/.aws/credentials`

If this file does not exist or you do not have a *secrets-user* IAM user, please refer to the previous lesson in this course 
[*Setting Up AWS CLI for the secrets user*](!TODO)

## Import the SDK and Create a Session
With the SDK installed in our environment and validated, we will need to import it before use and create an authenticated session with our *secrets-user* IAM account.

In [3]:
# import the library
import boto3

# boto responses are JSON, so we will want to use the json library a bit later
import json

# create a session with the profile for the secrets-user
session = boto3.Session(profile_name='secrets-user')

# output the region name as a light validation of creation
session.region_name

'us-east-2'

### Create a Client and List Secrets
Using the previously created session for the secrets-user, we will create a `'secretsmanager'` client and use it to list all the secrets in the account. Pay attention to the 'Name' in the output, as we will need that in the next step.

In [4]:
# create the client to access secrets manager, using the secrets-user session
## you do not have to pass the argument key `service_name`, but it helps readabilty 
client = session.client(service_name='secretsmanager')

# list out the secrets for the user and region
client.list_secrets()

{'SecretList': [{'ARN': 'arn:aws:secretsmanager:us-east-2:767398022036:secret:/dev/secrets-example/api-key-U2VcNW',
   'Name': '/dev/secrets-example/api-key',
   'Description': 'Mock API Key for demo',
   'LastChangedDate': datetime.datetime(2024, 2, 11, 13, 38, 14, 316000, tzinfo=tzlocal()),
   'Tags': [],
   'SecretVersionsToStages': {'c34f404d-040c-4062-98df-e05c42c22d97': ['AWSCURRENT']},
   'CreatedDate': datetime.datetime(2024, 2, 11, 13, 38, 14, 282000, tzinfo=tzlocal())}],
 'ResponseMetadata': {'RequestId': '3efb5bca-0339-4401-9fd4-61055b2bc762',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'x-amzn-requestid': '3efb5bca-0339-4401-9fd4-61055b2bc762',
   'content-type': 'application/x-amz-json-1.1',
   'content-length': '348',
   'date': 'Sun, 11 Feb 2024 18:52:38 GMT'},
  'RetryAttempts': 0}}

### Programmatically grab our secret.
Let's set a local variable for now and use it to get the secret, then expect the return value

In [5]:
# set up a local variable to hold our secret name (the name of the secret we see above from the output of `list_secrets()`)
secret_name = "/dev/secrets-example/api-key"

# use the method get_secret_value and pass the secret_name variable
response = client.get_secret_value(SecretId=secret_name)

# inspect the entire response object
response

{'ARN': 'arn:aws:secretsmanager:us-east-2:767398022036:secret:/dev/secrets-example/api-key-U2VcNW',
 'Name': '/dev/secrets-example/api-key',
 'VersionId': 'c34f404d-040c-4062-98df-e05c42c22d97',
 'SecretString': '{"API_KEY":"this-is-not-an-actual-key"}',
 'VersionStages': ['AWSCURRENT'],
 'CreatedDate': datetime.datetime(2024, 2, 11, 13, 38, 14, 313000, tzinfo=tzlocal()),
 'ResponseMetadata': {'RequestId': '71ad3ebb-00ac-44be-8e70-070cc8a8ecbc',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'x-amzn-requestid': '71ad3ebb-00ac-44be-8e70-070cc8a8ecbc',
   'content-type': 'application/x-amz-json-1.1',
   'content-length': '310',
   'date': 'Sun, 11 Feb 2024 18:53:29 GMT'},
  'RetryAttempts': 0}}

There is a lot of useful information here that you may want to use, but for this example we want to look at **'SecretString'** which should contain a key value pair that represents our api key name and value

In [6]:
# set the secret variable
secret = response['SecretString']

# inpect the type, which as you might expect, should be a 'str'
print(type(secret))

# the string is json, so we can easily parse it by converting it into a dict
secret = json.loads(secret)

# now we can access the API key
secret['API_KEY']

<class 'str'>


'this-is-not-an-actual-key'

### Options and Further Exploration

This toy example shows you the most rudimentary method of accessing a secret and making it accessible to your code.

Let's explore a few ideas around this:

#### Optional functional access

If you find yourself accessing secrets often, you can use the optional functional access pattern.

In [7]:
# this function is making several assumptions and comes with some caveats
    ## it does not have proper error checking (coming in the next lesson)
    ## it is not recommended to use this function in production
    ## it is also not optimized for performance
    ## it assumes the secrets-user and the region specified in that users credentials
    ## it assumes that the secret is stored as a string and not optional binary data

def get_secret(secret_name):

    session = boto3.session.Session(profile_name='secrets-user')
    client = session.client(service_name='secretsmanager')

    response = client.get_secret_value(
        SecretId=secret_name
    )
    
    if 'SecretString' in response:
        return response['SecretString']
    
# use the function
secret_name = '/dev/secrets-example/api-key' # was set earlier, but here again for readabilty
secret_from_function = get_secret(secret_name)

# inspect the return
secret_from_function

# convert to dict and output API key
secret_dict = json.loads(secret_from_function)
secret_dict['API_KEY']

'this-is-not-an-actual-key'

### Resources

#### Documentation and Developer Guides

[Boto3 Documentation](https://boto3.amazonaws.com/v1/documentation/api/latest/index.html)

[Boto3 Credentials Reference](https://boto3.amazonaws.com/v1/documentation/api/latest/guide/credentials.html)

[Boto3 Session Developer Guide](https://boto3.amazonaws.com/v1/documentation/api/latest/guide/session.html)

[Boto3 Secrets Manager Service Reference](https://boto3.amazonaws.com/v1/documentation/api/latest/reference/services/secretsmanager.html)


#### *Methods We Used*

[list_secrets](https://boto3.amazonaws.com/v1/documentation/api/latest/reference/services/secretsmanager/client/list_secrets.html)

[get_secret_value](https://boto3.amazonaws.com/v1/documentation/api/latest/reference/services/secretsmanager/client/get_secret_value.html)


#### *Feedback*

Feel free to send feedback however you like, it is always appreciated and helps me improve my work.

[Email]()

[Discord]()

[Other]()